# ⚔️ LevelUp AI — Backend Server

Run this notebook on **Google Colab with GPU** (Runtime → Change runtime type → T4 GPU)

After running all cells, you'll get a public URL like:
`https://xxxx-xxxx.ngrok-free.app`

Paste that URL into your frontend:  
`https://venkatkondaiahpalpu-levelup-ai.hf.space?api=YOUR_NGROK_URL`

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q fastapi uvicorn pyngrok python-dotenv
!pip install -q torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q transformers peft accelerate bitsandbytes sentencepiece
!pip install -q scikit-learn joblib elevenlabs openai-whisper
print('✅ Dependencies installed')

In [ ]:
# ── Cell 3: Clone your project ────────────────────────────────────────────────
import os

# Replace with your GitHub repo URL
REPO_URL = 'https://github.com/veeravenkatsaikondaiahpalpu-hue/levelup-ai.git'

if not os.path.exists('/content/levelup-ai'):
    !git clone {REPO_URL} /content/levelup-ai
else:
    !cd /content/levelup-ai && git pull

os.chdir('/content/levelup-ai')
print('✅ Project ready at /content/levelup-ai')

In [ ]:
# ── Cell 4: Configure secrets ─────────────────────────────────────────────────
# Paste your tokens here (or use Colab Secrets in the left sidebar 🔑)
import os

os.environ['HF_TOKEN']           = ''   # ← paste your HuggingFace token
os.environ['ELEVENLABS_API_KEY'] = ''   # ← paste your ElevenLabs key (optional)
os.environ['LEVELUP_LOAD_MODEL'] = 'true'
os.environ['LEVELUP_ADAPTER']    = 'models/final/levelup-qlora-cloud'  # local path
os.environ['WHISPER_MODEL']      = 'base'

print('✅ Environment configured')

In [ ]:
# ── Cell 5: Start backend + expose with ngrok ─────────────────────────────────
import subprocess, threading, time
from pyngrok import ngrok, conf

# Optional: set your ngrok authtoken for longer sessions
# Get free token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ''  # ← paste ngrok authtoken (optional but recommended)

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Start FastAPI in background
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

# Wait for model to load (this takes 2-3 minutes)
print('⏳ Loading model... (takes 2-3 min on first run)')
time.sleep(15)  # give uvicorn time to start

# Open ngrok tunnel
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

print(f'\n🚀 Backend is LIVE at:')
print(f'   {PUBLIC_URL}')
print(f'\n🌐 Open your frontend at:')
print(f'   https://venkatkondaiahpalpu-levelup-ai.hf.space?api={PUBLIC_URL}')
print(f'\n📖 API docs: {PUBLIC_URL}/docs')
print(f'\n⚠️  Keep this notebook running to stay live!')

In [ ]:
# ── Cell 6: Check server logs (run if something seems wrong) ──────────────────
import subprocess
result = subprocess.run(['curl', '-s', 'http://localhost:8000/health'], capture_output=True, text=True)
print('Health check:', result.stdout)

In [ ]:
# ── Cell 7: Keep alive (prevents Colab from disconnecting) ────────────────────
# Run this cell last — it prints a dot every 60s to prevent idle disconnect
import time
print('💓 Keep-alive running... (Ctrl+C to stop)')
while True:
    time.sleep(60)
    result = subprocess.run(['curl', '-s', 'http://localhost:8000/health'], capture_output=True, text=True)
    print(f'[{time.strftime("%H:%M")}] alive ✓  {result.stdout.strip()[:50]}')